In [13]:
#!/usr/bin/env python
# coding: utf-8
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from tqdm.notebook import tqdm
import pandas as pd
import os, sys, time
import numpy as np
from dlucj.classical import run_dice_addon



In [3]:



os.environ["OPENBLAS_CORETYPE"] = "generic"
os.environ["OPENBLAS_NUM_THREADS"] = "64"
os.environ["OMP_NUM_THREADS"] = "64"


In [4]:
basis_sets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']

active_spaces = pd.read_csv('../DDLUCJ_active_spaces_unfrozen.csv').dropna(axis=1)


df=pd.read_csv('energies.csv',index_col=0)


In [15]:
help(run_dice_addon)

Help on function run_dice_addon in module dlucj.classical:

run_dice_addon(
    structure,
    basis,
    active_space,
    sym,
    spin_sq,
    charge,
    n_froz,
    n_jobs=None
)



In [21]:

# All molecules are uncharged and closed-shell
open_shell = False
spin_sq = 0
energies = {}

#Iterate over basis sets
for b in basis_sets:
    energies[b] = {}
    # Iterate over DataFrame rows and find the structures in the directory
    for row in list(active_spaces.itertuples(index=False)):
        structure_dict = row._asdict()
        molname = structure_dict['molecule']
        molfroz = structure_dict['n_frozen']
        molelec = structure_dict['n_electrons']
        molorb = structure_dict['num_orbitals']

        # Find path
        if 'GDB' in molname:
            structpath = f"./structures/{molname}.xyz"
        else:
            structpath = glob(f"./structures/{molname}*.xyz")[0]

        if os.path.exists(structpath):
            print(structpath)
            print(b,molname,(molelec,molorb))
            print(molname)
            df = run_dice_addon(structpath, b, range(molorb), 'C1', 0, 0, molfroz,n_jobs=8)
            energies[b][molname] = df





# # Flatten the dictionary
# records = []
# for basis_set, molecules in energies.items():
#     for molecule, methods in molecules.items():
#         for method, energy in methods.items():
#             records.append((basis_set, molecule, method, energy))

# # Create DataFrame
# df = pd.DataFrame(records, columns=['Basis Set', 'Molecule', 'Method', 'Energy'])

# # Set MultiIndex
# # df.set_index(['Basis Set', 'Molecule', 'Method'], inplace=True)






# df.to_csv('energies.csv')









./structures/water183.xyz
STO-3G water (10, 7)
water
2S 0
S * (S+1) 0.0
Running HCI
SCF Energy -74.960552 Eh
HCI Energy -75.009478 Eh
./structures/ammonia157.xyz
STO-3G ammonia (10, 8)
ammonia
2S 0
S * (S+1) 0.0
Running HCI
SCF Energy -55.454318 Eh
HCI Energy -55.519894 Eh
./structures/methane50.xyz
STO-3G methane (10, 9)
methane
2S 0
S * (S+1) 0.0
Running HCI
SCF Energy -39.726623 Eh
HCI Energy -39.805968 Eh
./structures/formaldehyde138.xyz
STO-3G formaldehyde (16, 12)
formaldehyde
2S 0
S * (S+1) 0.0
Running HCI
SCF Energy -112.353755 Eh
HCI Energy -112.500988 Eh
./structures/ethylene42.xyz
STO-3G ethylene (16, 14)
ethylene
2S 0
S * (S+1) 0.0
Running HCI
SCF Energy -77.071233 Eh
HCI Energy -77.233917 Eh
./structures/ethane28.xyz
STO-3G ethane (18, 16)
ethane
2S 0
S * (S+1) 0.0
Running HCI
SCF Energy -78.305816 Eh
HCI Energy -78.448955 Eh
./structures/methanol22.xyz
STO-3G methanol (18, 14)
methanol
2S 0
S * (S+1) 0.0
Running HCI
SCF Energy -113.544759 Eh
HCI Energy -113.660765 Eh
./st

In [35]:
stackeddf=[]
for k,v in energies.items():
    for k1,v1 in v.items():
        df = v1.copy()
        df['Name']=k1
        stackeddf.append(df)

In [37]:
pd.concat(stackeddf).to_csv('energies.csv')